# DD-03 · App process launch: Zygote fork to Application.onCreate

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/DD-03.ipynb)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: DD-03
title: "App process launch: Zygote fork to Application.onCreate"
track: android
date: 2026-10-17
code: ""
status: partial  # not-started | in-progress | partial | done
score: ""
insight: "Cold-start time is how long the main thread is busy before the first frame: Zygote forks a warm process, ActivityThread.main starts the Looper, then Application, the ContentProviders (before Application.onCreate) and the Activity all run on the main thread, so doing less there is how an app starts faster."
tags: [DD-03, android, app-launch, zygote, looper, cold-start, content-provider, perfetto, probe-lab]
pending:
  - "MQ31 and MQ32 in my own words"
  - "English (overhead, redundant) and the trade-off matrix drill"
  - "Block A: DSA timed set (LC 410, LC 1482, LC 162)"
  - "Optional: composition tracing to see what fills the 27 ms before the first measure"
```

## Goal
Know exactly what happens between tapping an icon and the first frame, and why app launch is fast (Zygote) and where startup time is spent (the main thread).

## The story: a cold start in 9 steps
```text
1  You tap the icon
2  Launcher asks system_server to start the app            (a Binder call)
3  system_server finds NO process → shows a placeholder window (splash) at once
4  system_server asks Zygote for a process                 (the Zygote socket, DD-02)
5  The child runs ActivityThread.main()                    → Looper.prepareMainLooper, attach, Looper.loop()
6  system_server answers with the app's details            (bindApplication)
7  The app creates the Application, starts the ContentProviders, runs Application.onCreate
8  system_server asks the app to launch the Activity       → onCreate, onStart, onResume
9  The first frame is drawn                                → "Displayed ... +NNNms" in logcat
```
A **cold start** has no process, so all nine steps happen. A **warm start** has a process but no Activity (steps 8 and 9). A **hot start** only resumes an Activity still in memory.

## First half (steps 1 to 5)
- **The tap:** the Launcher is an ordinary app. Its `startActivity` travels through **Binder** to `system_server` (ActivityManager). If the app has no process, `system_server` shows a **starting window** straight away, built from the theme's `windowBackground` and icon. That is why a start can *feel* instant, and why that theme matters.
- **The fork:** `system_server` writes to the **Zygote socket**; Zygote calls `fork()`. The child shares the preloaded framework (copy-on-write) and **specialises itself**: it changes its user ID to the app's own (`u0_a125` is Linux user 10125), which is the start of the app sandbox. This takes milliseconds because no JVM is started from scratch.
- **`ActivityThread.main()`** is the app's real `main`. It (1) `Looper.prepareMainLooper()`, makes the main thread's **mailbox**; (2) `attach`, calls `system_server` over Binder: "I'm alive"; (3) `Looper.loop()`, an endless loop: wait for a message, run it, repeat. From here on, everything (lifecycle, clicks, drawing) arrives as a message on the main thread. One slow message makes the next ones wait: a dropped frame, and eventually an ANR.

## Second half (steps 6 to 9)
- **`bindApplication`:** arrives on a **Binder thread**, which only **posts a message** into the main Looper. The main thread does the work. The standard handoff: *Binder thread → main thread*.
- **The order that decides cold start** (all on the main thread):
  ```text
  1. Application object created      (constructor)
  2. attachBaseContext(...)
  3. every ContentProvider.onCreate() (yours AND every library's)
  4. Application.onCreate()           (your code)
  ```
  Providers run **before** your `Application.onCreate`. Many libraries used a ContentProvider to initialise themselves, and each one added startup time before your own code even began. Jetpack **App Startup** replaces many providers with one that initialises in dependency order.
- **The Activity:** another message on the main Looper. `onCreate` (where `setContent { }` or layout inflation happens), `onStart`, `onResume`.
- **The first frame:** at the next VSYNC the system measures, lays out and draws; `system_server` then logs `Displayed pkg/.Activity: +NNNms`.

## The rule
> **Cold-start time is how long the main thread is busy before the first frame.** So: lighter providers, a lighter `Application.onCreate`, a lighter `Activity.onCreate`; defer what can wait; move the rest to a background thread.

## MQ31 and MQ32
**MQ31 · How does tapping an app icon launch the app?** The nine steps above.
**MQ32 · How does the Launcher know the installed apps?** `PackageManager` in `system_server` scanned every APK at boot (the `pms_*` events timed in DD-02) and keeps the list, updated on every install and removal. The Launcher asks for every activity that declares `MAIN` + `LAUNCHER` (through the `LauncherApps` service or `queryIntentActivities`) and gets a callback when an app is installed or removed.
✏️ Write both in my own words (2 lines each).

## The commands, explained
```bash
adb shell am start -S -W dev.builtbyswap.kaizen/.MainActivity
```
| Part | Meaning |
|---|---|
| `adb shell` | Run the command **on the phone or emulator** |
| `am` | **Activity Manager** tool: talks to the service that starts apps |
| `start` | Start an Activity |
| `-S` | **S**top the app first, which forces a **cold** start |
| `-W` | **W**ait for the launch to finish, then print `LaunchState` and `TotalTime` |
| `package/.Class` | Which Activity (the dot means "inside the app's own package") |

Without `-S` an app that is already running just comes to the front (a warm or hot start), so the timing means something else. Without `-W` the command returns at once.

## My data: the fork, from my emulator

In [ ]:
ps = """
u0_a125      19878  1047   15681904 188160 0                   0 S dev.builtbyswap.kaizen
root          1047     1   15363768  80808 0                   0 S zygote64
"""

rows = [line.split() for line in ps.strip().splitlines()]
procs = {int(r[1]): (int(r[2]), r[-1], r[0]) for r in rows}          # pid -> (parent pid, name, user)

for pid, (ppid, name, user) in procs.items():
    print(f"{name}: PID {pid}, parent PID {ppid}, user {user}")

app_pid = next(pid for pid, (_, name, _) in procs.items() if name.endswith("kaizen"))
zygote_pid = next(pid for pid, (_, name, _) in procs.items() if name == "zygote64")
assert procs[app_pid][0] == zygote_pid                                 # the app's parent is Zygote
print("the app's parent is zygote64 (PID", zygote_pid, "): the app was forked from Zygote")

uid = 10000 + int(procs[app_pid][2].split("_a")[1])                    # u0_a125 -> 10125
print("Linux user id of the app:", uid)
assert uid == 10125

## My data: the cold-start time
`am start -S -W` gave `TotalTime: 294`, then `289`. The `Displayed dev.builtbyswap.kaizen/.MainActivity for user 0: +289ms` line in logcat (written by `ActivityTaskManager`, not by the app: the system times the launch from outside) matched the second run exactly: two tools reporting one moment.

**One run is not a measurement.** Startup time varies with caches, CPU frequency and background work. Take **five runs and use the median**, and compare future changes (a new library, an SDK, a Baseline Profile) against that baseline:
```bash
for i in 1 2 3 4 5; do adb shell am start -S -W dev.builtbyswap.kaizen/.MainActivity | grep TotalTime; done
```

In [ ]:
from statistics import median

def summarize(runs_ms):
    return {"runs": len(runs_ms), "median": median(runs_ms), "min": min(runs_ms), "max": max(runs_ms)}

def regression(baseline_ms, new_ms, allowed=0.10):
    """True if the new build is more than 10% slower than the baseline."""
    return new_ms > baseline_ms * (1 + allowed)

mine = [294, 289]                        # my two runs; add the other three when measured
baseline = summarize(mine)
print(baseline)
print("a 330 ms build is a regression:", regression(baseline["median"], 330))
print("a 300 ms build is a regression:", regression(baseline["median"], 300))
assert baseline["median"] == 291.5
assert regression(baseline["median"], 330) is True and regression(baseline["median"], 300) is False

## Probe lab: my results
A throwaway project (`AppLaunchLab`, never pushed) with log lines in the `Application` constructor, `attachBaseContext`, a `ContentProvider.onCreate`, `Application.onCreate` and the Activity's `onCreate`, `onStart` and `onResume`. I added `Thread.sleep(500)` in three places and read the timestamps.

**The order was the same every time, all on the thread `main`:** `1 constructor → 2 attachBaseContext → 3 ContentProvider.onCreate → 4 Application.onCreate → 5 Activity.onCreate → 6 onStart → 7 onResume`. The provider ran before `Application.onCreate`.

| Experiment | Lines 1 to 7 took | The biggest gap | Everything after `Application.onCreate` |
|---|---|---|---|
| 1 · `sleep(500)` in `ProbeProvider.onCreate` | **567 ms** | **502 ms**, between lines 2 and 3 | 51 ms |
| 2 · `sleep(500)` in `ProbeApp.onCreate` | **561 ms** | **515 ms**, between lines 3 and 4 | 45 ms |
| 3 · the sleep on a background thread | **50 ms** | none (the biggest is 19 ms) | 38 ms |

**What it proves:** my normal startup work is about 50 ms, and a sleep on the main thread adds about +515 ms. The main thread is a single line, so one slow step delays every step after it. Moving the work off the main thread removed the delay, **but** it was only *sleeping*, and sleeping uses no CPU. **Waiting overlaps for free; computing does not** (AND-01): real CPU work on a background thread competes for cores, and work the first screen needs must still be waited for. (`TotalTime` over three runs: 546, 554 and 558 ms for one of the sleep experiments; the baseline median is still to be measured.)

In [ ]:
import re

def to_ms(t):
    h, m, s = t.split(":")
    return (int(h) * 3600 + int(m) * 60) * 1000 + round(float(s) * 1000)

def analyse(log):
    points = [(int(m.group(2)), to_ms(m.group(1))) for m in re.finditer(r"(\d\d:\d\d:\d\d\.\d+) D\s+(\d) ", log)]
    gaps = {f"{a}->{b}": tb - ta for (a, ta), (b, tb) in zip(points, points[1:])}
    after_app = points[-1][1] - dict(points)[4]
    return points[-1][1] - points[0][1], max(gaps.items(), key=lambda g: g[1]), after_app

exp1 = """
15:50:19.008 D  1 Application constructor [main]
15:50:19.008 D  2 attachBaseContext [main]
15:50:19.510 D  3 ContentProvider.onCreate [main]
15:50:19.524 D  4 Application.onCreate [main]
15:50:19.541 D  5 Activity.onCreate [main]
15:50:19.568 D  6 Activity.onStart [main]
15:50:19.575 D  7 Activity.onResume [main]
"""
exp2 = """
15:51:21.937 D  1 Application constructor [main]
15:51:21.937 D  2 attachBaseContext [main]
15:51:21.938 D  3 ContentProvider.onCreate [main]
15:51:22.453 D  4 Application.onCreate [main]
15:51:22.477 D  5 Activity.onCreate [main]
15:51:22.493 D  6 Activity.onStart [main]
15:51:22.498 D  7 Activity.onResume [main]
"""
exp3 = """
15:51:46.547 D  1 Application constructor [main]
15:51:46.547 D  2 attachBaseContext [main]
15:51:46.548 D  3 ContentProvider.onCreate [main]
15:51:46.559 D  4 Application.onCreate [main]
15:51:46.572 D  5 Activity.onCreate [main]
15:51:46.591 D  6 Activity.onStart [main]
15:51:46.597 D  7 Activity.onResume [main]
"""
for name, log in (("1 sleep in the provider", exp1), ("2 sleep in Application.onCreate", exp2), ("3 sleep on a background thread", exp3)):
    total, biggest, after_app = analyse(log)
    print(f"{name}: lines 1 to 7 took {total} ms, biggest gap {biggest[0]} = {biggest[1]} ms, after Application.onCreate {after_app} ms")

assert analyse(exp1)[0] == 567 and analyse(exp1)[1] == ("2->3", 502)
assert analyse(exp2)[0] == 561 and analyse(exp2)[1] == ("3->4", 515)
assert analyse(exp3)[0] == 50 and analyse(exp3)[1][1] < 25

## Perfetto trace: my cold start, millisecond by millisecond
**How it was recorded.** Android Studio's **Capture System Activities** task with "from process start" answered *Task could not be launched on startup* (cause not diagnosed), so I recorded from the command line instead: `adb shell perfetto -o /data/misc/perfetto-traces/cold.pftrace -t 10s sched freq idle am wm gfx view binder_driver hal dalvik input res memory`, ran `am start -S -W` in a second tab, then `adb pull` the file (6.7 MB). The numbers below come from querying that file with Perfetto's trace processor, the same engine as the browser UI.

**It confirmed my earlier `ps` output:** `zygote64` is PID 1047 and `system_server` is PID 1766, the PID on the `Displayed` log line, so that line really is written by the system and not by the app. The app was PID 22126, on a **debuggable** build.

**The timeline** (ms from the moment the launch was requested; the system measured **159.4 ms** in total for this cold start):

| Landmark | Starts (ms) | Lasts (ms) | In |
|---|---|---|---|
| Launch requested (`launchingActivity`) | 0 | 159.4 in total | system_server |
| `launching component` (the intent is resolved) | 26.0 | | system_server |
| `Start proc` (asks Zygote for a process) | 26.9 | 11.9 | system_server |
| `forkApp` (the fork itself) | 27.6 | 11.1 | zygote64 |
| `PostFork` (the child starts running) | 50.7 | 1.1 | the app |
| `ActivityThreadMain` | 54.0 | 4.5 | the app |
| `bindApplication` | 58.8 | 22.3 | the app |
| `activityStart` (with `performCreate:MainActivity` 7.6 ms) | 81.5 | 14.5 | the app |
| `activityResume` | 96.5 | 17.7 | the app |
| first real frame: `Choreographer#doFrame` | 114.7 | 43.1 | the app |
| launch marked complete (cold) | 159.5 | | system_server |

**Inside the frame:** `measure` 2.2 ms, `relayoutWindow` 3.1 ms, `layout` 2.6 ms, `draw` 1.7 ms. That is about 9.5 ms of the classic work, and about **27 ms before `measure` even begins**.

In [ ]:
launch_ms = 159.43
phases = [                                                              # (name, start, end) in ms
    ("before the app runs (system_server, fork, child setup)", 0.00, 50.70),
    ("process setup (PostFork, ZygoteInit, ActivityThreadMain)", 50.70, 58.84),
    ("bindApplication",                                         58.84, 81.17),
    ("activity lifecycle (create, start, resume)",              81.17, 114.66),
    ("first frame (doFrame + traversal)",                       114.66, 157.80),
    ("until the launch is marked complete",                     157.80, 159.43),
]
total = 0
for name, a, b in phases:
    total += b - a
    print(f"{b - a:6.1f} ms  {100 * (b - a) / launch_ms:4.0f}%  {name}")
assert abs(total - launch_ms) < 0.01                                    # the phases cover the whole launch
assert phases[0][2] / launch_ms > 0.3                                   # a third happens before the app's code runs

# the main thread's states inside the four windows where the app runs (ms)
states = {"Running": 4.73 + 18.39 + 17.45 + 34.24, "Sleeping": 0.57 + 3.14 + 13.56 + 8.12,
          "Runnable": 2.39 + 0.03 + 0.56 + 0.24 + 1.21 + 0.54 + 0.23, "Disk I/O wait": 0.08 + 0.54 + 0.02}
busy = sum(states.values())
for state, ms in states.items():
    print(f"{state:14s} {ms:6.1f} ms  {100 * ms / busy:4.0f}%")
assert abs(busy - 106.03) < 0.1
assert states["Running"] / busy > 0.7

**What the trace says**
1. **A third of the cold start (50.7 of 159 ms) happens before the app's code runs at all:** the system resolves the intent (26 ms), Zygote forks (11 ms) and the child is set up. App code cannot change that part.
2. **The app's own `Application` work is tiny.** `makeApplication` took 0.84 ms. `bindApplication` (22 ms) is mostly the cost of opening the app's code (`OpenDexFilesFromOat` 8.6 ms, already compiled ahead of time: `status=up-to-date filter=speed-profile`).
3. **The biggest piece is the first frame: 43 ms, 27% of the start.** Only about 9.5 ms is measure, layout and draw. About 27 ms come *before* `measure`; the small slices visible there are class loading (`AndroidComposeView`, kotlinx.coroutines classes) and Binder calls, so this is where Compose builds its first UI. I cannot see *which composables* without composition tracing.
4. **Working vs waiting** (106 ms of main-thread time): about **71% Running** (real CPU work), **24% Sleeping** (waiting), **5% Runnable** (ready, waiting for a CPU core), under 1% disk I/O. The waiting is concentrated in the activity lifecycle (41% of that phase, 13.6 of 32.8 ms): five Binder calls to `system_server` inside `activityResume` (2.4 + 1.0 + 3.3 + 1.2 + 1.4 = 9.3 ms), each one a main-thread wait.
5. **159 ms here vs 289 to 294 ms in my earlier `am start -S -W` runs:** different runs of the same app on the same device. A likely reason is the state of the compiled code (this trace shows it fully compiled from a profile), but I have not proved it. **Measure five runs and take the median** now.

**Query it yourself** (ui.perfetto.dev → *Query (SQL)* page, with the trace open):
```sql
INCLUDE PERFETTO MODULE android.startup.startups;
select startup_id, dur / 1e6 as ms, package, startup_type from android_startups;
```
```sql
select ts, dur, name from slice
where name like 'bindApplication%' or name like 'activityStart%' or name like 'activityResume%'
   or name like 'Start proc%' or name like 'Choreographer#doFrame%'
order by ts;
```
**Optional next step: composition tracing.** Adding the Compose runtime-tracing dependency (the "Composition tracing" page on developer.android.com names it and the current version) makes composable function names appear inside the trace, which would show what fills the 27 ms before `measure`.

## Bug diary
| Error or symptom | Cause | Fix |
|---|---|---|
| I called one run a measurement | Startup time varies from run to run (294 then 289 ms; later a trace showed 159 ms) | Five runs, take the median |
| `Task could not be launched on startup` in the Android Studio Profiler | Not diagnosed (the app was already running under the debugger from Android Studio) | Recorded with the command-line Perfetto route instead |

## Quiz and mistakes
Right: every `adb` flag explained back to me (`-S`, `-W`); the PPID check proved the fork; the probe-lab predictions (a sleep on the main thread adds about 500 ms; off the main thread it adds none). No quiz today.

## English and thinking
Words: **overhead** (extra cost), **redundant** (extra; unnecessary, or kept for safety). Grammar: past simple vs present perfect. Thinking model: **trade-off matrix**.
✏️ Three sentences with both words. ✏️ Drill: a 3×4 matrix for one decision from today (for example how to initialise an SDK: eagerly in `Application.onCreate`, lazily on first use, on a background thread, or with App Startup, scored against cold-start time, simplicity, correctness and testability).

## In my own words
✏️ Why does a ContentProvider add to cold-start time even though my own code has not started yet?

## Next
**AI-03** continues on Sunday (the backward pass step by step). **PRJ-03:** Postgres.